In [1]:
# ============================================================
# Fine-Tuning-PS-LLM
# Notebook: 03_evaluate.ipynb
# Section: Fix Environment
# Purpose: Upgrade torchao for PEFT compatibility.
# ============================================================

!pip install -q -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 49.1 MB/s eta 0:00:00


In [2]:
# ============================================================
# Mount Google Drive
# ============================================================

from pathlib import Path

if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Google Drive is already mounted.")

Mounted at /content/drive


In [3]:
# ============================================================
# Fine-Tuning-PS-LLM
# Notebook: 03_evaluate.ipynb
# Section: Project Paths
# Purpose: Configure project directories.
# ============================================================

from pathlib import Path
import sys

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Colab Notebooks/Fine-Tuning-PS-LLM"
)

sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT)

/content/drive/MyDrive/Colab Notebooks/Fine-Tuning-PS-LLM


In [4]:
# ============================================================
# Fine-Tuning-PS-LLM
# Notebook: 03_evaluate.ipynb
# Section: Import Libraries
# Purpose: Import required libraries.
# ============================================================

import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)

from peft import (
    PeftModel,
)

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)

from utils.dataset import (
    load_data,
    merge_data,
    split_data,
)

from utils.inference import (
    predict_quote,
    parse_response,
)

In [5]:
# ============================================================
# Fine-Tuning-PS-LLM
# Notebook: 03_evaluate.ipynb
# Section: Load Prompt Template
# Purpose: Load prompt P01 used during fine-tuning.
# ============================================================

P01_PROMPT = (
    PROJECT_ROOT
    / "prompts"
    / "P01_Zero_Shot_Closed_Coding.txt"
).read_text(
    encoding="utf-8"
)

print(P01_PROMPT[:800])

You are a software engineering researcher specializing in qualitative analysis of psychological safety in software development teams.

Your task is to perform qualitative coding of the quotes provided below by identifying and classifying psychological safety challenges described in each quote.

The goal is to identify challenges related to psychological safety, such as interpersonal risk-taking, learning anxiety, defensive behaviors, or resistance to change.

---

Context for Classification

Psychological safety refers to a person's perception that they can take interpersonal risks in a team environment without fear of negative consequences such as embarrassment, rejection, or punishment.

---

Unit of Analysis

Each quote represents a single unit of analysis.

Even if multiple psychologic


In [6]:
from pathlib import Path
import os

project = Path("/content/drive/MyDrive/Colab Notebooks/Fine-Tuning-PS-LLM")

print("Exists:", project.exists())
print()

print("Contents:")
for item in sorted(os.listdir(project)):
    print(item)

Exists: True

Contents:
.gitignore
LICENSE
README.md
archive
data
docs
figures
models
notebooks
outputs
prompts
requirements.txt
results
utils


In [7]:
# ============================================================
# Fine-Tuning-PS-LLM
# Notebook: 03_evaluate.ipynb
# Section: Load Test Dataset
# Purpose: Load the raw datasets and recreate the test split.
# ============================================================

quotes, gold = load_data(PROJECT_ROOT)

dataset = merge_data(
    quotes,
    gold,
)

_, _, test_df = split_data(
    dataset,
)

print(f"Test samples: {len(test_df)}")

test_df.head()

Test samples: 12


,quote_id,quote_text,gold_category
0,Quote 71,Should the developer shoot down the idea becau...,Disagreeing with suggestions/ideas
1,Quote 57,I am on the product side and have an agile tea...,Expressing concerns
2,Quote 84,"I have a strange situation at work, where a co...",Seeking help
3,Quote 110,To do this I have discussion with related team...,Disagreeing with suggestions/ideas
4,Quote 27,How to influence a scrum team to do something ...,Recommending changes


In [8]:
# ============================================================
# Fine-Tuning-PS-LLM
# Notebook: 03_evaluate.ipynb
# Section: Configure Evaluation Model
# Purpose: Select Baseline, SFT, or DPO.
# ============================================================

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

MODEL_TYPE = "dpo"
# Available:
# "baseline"
# "sft"
# "dpo"

if MODEL_TYPE == "baseline":

    USE_LORA = False
    ADAPTER_PATH = None

elif MODEL_TYPE == "sft":

    USE_LORA = True
    ADAPTER_PATH = (
        PROJECT_ROOT
        / "models"
        / "sft_qwen"
    )

elif MODEL_TYPE == "dpo":

    USE_LORA = True
    ADAPTER_PATH = (
        PROJECT_ROOT
        / "models"
        / "dpo_qwen"
    )

else:

    raise ValueError("Unknown MODEL_TYPE")

print(f"Model Type : {MODEL_TYPE}")

if ADAPTER_PATH is not None:
    print(f"Adapter    : {ADAPTER_PATH}")

Model Type : dpo
Adapter    : /content/drive/MyDrive/Colab Notebooks/Fine-Tuning-PS-LLM/models/dpo_qwen


In [10]:
# ============================================================
# Fine-Tuning-PS-LLM
# Notebook: 03_evaluate.ipynb
# Section: Load Evaluation Model
# Purpose: Load the base model and apply the selected adapter
#          (Baseline, SFT, or DPO).
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16,
)

print("Base model loaded successfully.")

from peft import PeftModel

if USE_LORA:

    model = PeftModel.from_pretrained(
        model,
        ADAPTER_PATH,
    )

    print(f"{MODEL_TYPE.upper()} adapter loaded successfully.")

else:

    print("Running Base Qwen model.")

model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Base model loaded successfully.
DPO adapter loaded successfully.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=4, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=4, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear(in_feat

In [11]:
# ============================================================
# Fine-Tuning-PS-LLM
# Notebook: 03_evaluate.ipynb
# Section: Load LoRA Adapter
# Purpose: Load adapter only when required.
# ============================================================

from peft import PeftModel

if USE_LORA:

    model = PeftModel.from_pretrained(
        model,
        ADAPTER_PATH,
    )

    print(f"{MODEL_TYPE.upper()} adapter loaded successfully.")

else:

    print("Running Base Qwen model.")

model.eval()

DPO adapter loaded successfully.


/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:622: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.base_model.model.model.layers.1.self_attn.q_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.layers.1.self_attn.q_proj.lora_B.default.weight', 'base_model.model.base_model.model.model.layers.1.self_attn.v_proj.

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): PeftModelForCausalLM(
      (base_model): LoraModel(
        (model): Qwen2ForCausalLM(
          (model): Qwen2Model(
            (embed_tokens): Embedding(151936, 1536)
            (layers): ModuleList(
              (0-27): 28 x Qwen2DecoderLayer(
                (self_attn): Qwen2Attention(
                  (q_proj): lora.Linear(
                    (base_layer): Linear(in_features=1536, out_features=1536, bias=True)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.05, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(in_features=1536, out_features=4, bias=False)
                    )
                    (lora_B): ModuleDict(
                      (default): Linear(in_features=4, out_features=1536, bias=False)
                    )
                    (lora_embedding_A): ParameterDict()
                    (l

In [12]:
# ============================================================
# Fine-Tuning-PS-LLM
# Notebook: 03_evaluate.ipynb
# Section: Run Inference
# Purpose: Generate predictions for the test dataset.
# ============================================================

predictions = []

for i, (_, row) in enumerate(test_df.iterrows(), start=1):

    print(f"{i}/{len(test_df)} : {row['quote_id']}")

    response = predict_quote(
        model=model,
        tokenizer=tokenizer,
        prompt_template=P01_PROMPT,
        quote_id=row["quote_id"],
        quote_text=row["quote_text"],
    )



    try:

        parsed = parse_response(response)

        predicted_category = parsed.get(
            "category",
            None,
        )

    except Exception as e:

        print(f"\nFailed on {row['quote_id']}")
        print(response)
        print("-" * 80)

        predicted_category = None

    predictions.append(
        {
            "quote_id": row["quote_id"],
            "gold_category": row["gold_category"],
            "predicted_category": predicted_category,
            "raw_response": response,
        }
    )

print(f"\nFinished inference on {len(predictions)} quotes.")

1/12 : Quote 71
2/12 : Quote 57
3/12 : Quote 84
4/12 : Quote 110
5/12 : Quote 27
6/12 : Quote 12
7/12 : Quote 54
8/12 : Quote 05
9/12 : Quote 89
10/12 : Quote 48
11/12 : Quote 43
12/12 : Quote 95

Finished inference on 12 quotes.


In [13]:
# ============================================================
# Fine-Tuning-PS-LLM
# Notebook: 03_evaluate.ipynb
# Section: Build Results DataFrame
# Purpose: Convert predictions into a pandas DataFrame.
# ============================================================

results_df = pd.DataFrame(predictions)

print(results_df)

print()

print(results_df["predicted_category"])

     quote_id                       gold_category  \
0    Quote 71  Disagreeing with suggestions/ideas   
1    Quote 57                 Expressing concerns   
2    Quote 84                        Seeking help   
3   Quote 110  Disagreeing with suggestions/ideas   
4    Quote 27                Recommending changes   
5    Quote 12  Disagreeing with suggestions/ideas   
6    Quote 54                 Expressing concerns   
7    Quote 05                Recommending changes   
8    Quote 89                Recommending changes   
9    Quote 48  Disagreeing with suggestions/ideas   
10   Quote 43         Drawing attention to errors   
11   Quote 95                 Expressing concerns   

             predicted_category  \
0   Drawing attention to errors   
1           Expressing concerns   
2           Expressing concerns   
3           Expressing concerns   
4           Expressing concerns   
5   Drawing attention to errors   
6           Expressing concerns   
7           Expressing concern

In [14]:
# ============================================================
# Fine-Tuning-PS-LLM
# Notebook: 03_evaluate.ipynb
# Section: Classification Metrics
# Purpose: Compute evaluation metrics.
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)

# ------------------------------------------------------------
# Remove failed predictions (if any)
# ------------------------------------------------------------

evaluation_df = results_df.dropna(
    subset=["predicted_category"]
).reset_index(drop=True)

print(f"Evaluated samples: {len(evaluation_df)}")
print()

# ------------------------------------------------------------
# Accuracy
# ------------------------------------------------------------

accuracy = accuracy_score(
    evaluation_df["gold_category"],
    evaluation_df["predicted_category"],
)

print(f"Accuracy: {accuracy:.4f}")
print()

# ------------------------------------------------------------
# Classification Report
# ------------------------------------------------------------

print(
    classification_report(
        evaluation_df["gold_category"],
        evaluation_df["predicted_category"],
        zero_division=0,
    )
)

# ------------------------------------------------------------
# Confusion Matrix
# ------------------------------------------------------------

cm = confusion_matrix(
    evaluation_df["gold_category"],
    evaluation_df["predicted_category"],
)

print("Confusion Matrix:")
print(cm)

Evaluated samples: 12

Accuracy: 0.2500

                                    precision    recall  f1-score   support

Disagreeing with suggestions/ideas       0.00      0.00      0.00         4
       Drawing attention to errors       0.00      0.00      0.00         1
               Expressing concerns       0.33      1.00      0.50         3
              Recommending changes       0.00      0.00      0.00         3
                      Seeking help       0.00      0.00      0.00         1

                          accuracy                           0.25        12
                         macro avg       0.07      0.20      0.10        12
                      weighted avg       0.08      0.25      0.12        12

Confusion Matrix:
[[0 3 1 0 0]
 [0 0 1 0 0]
 [0 0 3 0 0]
 [0 0 3 0 0]
 [0 0 1 0 0]]


In [15]:
# ============================================================
# Save Results
# ============================================================

RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / MODEL_TYPE
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

results_path = (
    RESULTS_DIR
    / f"{MODEL_TYPE}_predictions.csv"
)

results_df.to_csv(
    results_path,
    index=False,
)

print()
print("Results saved to:")
print(results_path)

print()
print(f"Total predictions: {len(results_df)}")
print(f"Saved {len(results_df)} predictions for {MODEL_TYPE}.")


Results saved to:
/content/drive/MyDrive/Colab Notebooks/Fine-Tuning-PS-LLM/results/dpo/dpo_predictions.csv

Total predictions: 12
Saved 12 predictions for dpo.


In [16]:
print(type(model))

<class 'peft.peft_model.PeftModelForCausalLM'>


In [ ]:
print(len(predictions))

12


In [ ]:
print(len(predictions))

results_df = pd.DataFrame(predictions)

print(len(results_df))

results_df.head()

12
12


,quote_id,gold_category,predicted_category,raw_response
0,Quote 71,Disagreeing with suggestions/ideas,Drawing attention to errors,"```json\n[\n {\n ""id_quote"": ""Quote ..."
1,Quote 57,Expressing concerns,Expressing concerns,"```json\n[\n {\n ""id_quote"": ""Quote ..."
2,Quote 84,Seeking help,Expressing concerns,"```json\n[\n {\n ""id_quote"": ""Quote ..."
3,Quote 110,Disagreeing with suggestions/ideas,Expressing concerns,"```json\n[\n {\n ""id_quote"": ""Quote ..."
4,Quote 27,Recommending changes,Expressing concerns,"```json\n[\n {\n ""id_quote"": ""Quote ..."


In [ ]:
import importlib
import utils.inference

importlib.reload(utils.inference)

from utils.inference import (
    predict_quote,
    parse_response,
)

print("Inference module reloaded.")

Inference module reloaded.


In [18]:
from pathlib import Path
import json

for ckpt in [
    PROJECT_ROOT / "models" / "sft_qwen" / "checkpoint-33" / "trainer_state.json",
    PROJECT_ROOT / "models" / "dpo_qwen" / "checkpoint-14" / "trainer_state.json",
]:
    print("=" * 60)
    print(ckpt)

    with open(ckpt, "r") as f:
        state = json.load(f)

    print("Best metric :", state.get("best_metric"))
    print("Global step :", state.get("global_step"))

    if "log_history" in state:
        print("Last log:")
        print(state["log_history"][-1])

/content/drive/MyDrive/Colab Notebooks/Fine-Tuning-PS-LLM/models/sft_qwen/checkpoint-33/trainer_state.json
Best metric : 0.0019636068027466536
Global step : 33
Last log:
{'epoch': 2.782608695652174, 'eval_loss': 0.0019636068027466536, 'eval_mean_token_accuracy': 1.0, 'eval_runtime': 3.4191, 'eval_samples_per_second': 3.51, 'eval_steps_per_second': 0.585, 'step': 33}
/content/drive/MyDrive/Colab Notebooks/Fine-Tuning-PS-LLM/models/dpo_qwen/checkpoint-14/trainer_state.json
Best metric : None
Global step : 14
Last log:
{'entropy': 0.9190862841076322, 'epoch': 1.4615384615384617, 'grad_norm': 0.5859375, 'learning_rate': 1.7857142857142859e-06, 'logits/chosen': 0.061839864165462215, 'logits/rejected': 0.043342623584018976, 'logps/chosen': -378.5307269626194, 'logps/rejected': -376.75509474012586, 'loss': 0.6986292362213135, 'mean_token_accuracy': 0.7927834524048699, 'num_tokens': 75752.0, 'rewards/accuracies': 0.3611111111111111, 'rewards/chosen': 0.0028149497076002364, 'rewards/margins': -